In [1]:
import xarray as xr
from pathlib import Path
from xhistogram.xarray import histogram as xhist
import numpy as np
import matplotlib.pyplot as plt

In [2]:
stores = sorted(Path("../data/tracks").glob("Parcels_run_*_*.zarr"))
stores

[PosixPath('../data/tracks/Parcels_run_1234_2022-06-01T00:00:00.zarr'),
 PosixPath('../data/tracks/Parcels_run_1234_2022-07-01T00:00:00.zarr'),
 PosixPath('../data/tracks/Parcels_run_1234_2022-08-01T00:00:00.zarr'),
 PosixPath('../data/tracks/Parcels_run_1234_2022-09-01T00:00:00.zarr'),
 PosixPath('../data/tracks/Parcels_run_1234_2022-10-01T00:00:00.zarr'),
 PosixPath('../data/tracks/Parcels_run_1234_2022-11-01T00:00:00.zarr'),
 PosixPath('../data/tracks/Parcels_run_1234_2022-12-01T00:00:00.zarr'),
 PosixPath('../data/tracks/Parcels_run_1234_2023-01-01T00:00:00.zarr'),
 PosixPath('../data/tracks/Parcels_run_1234_2023-02-01T00:00:00.zarr'),
 PosixPath('../data/tracks/Parcels_run_1234_2023-03-01T00:00:00.zarr'),
 PosixPath('../data/tracks/Parcels_run_1234_2023-04-01T00:00:00.zarr'),
 PosixPath('../data/tracks/Parcels_run_1234_2023-05-01T00:00:00.zarr'),
 PosixPath('../data/tracks/Parcels_run_1234_2023-06-01T00:00:00.zarr'),
 PosixPath('../data/tracks/Parcels_run_1234_2023-07-01T00:00:00.

In [3]:
ds_list = [xr.open_zarr(s).assign(stores=s.name) for s in stores]
ds = xr.concat(ds_list,dim='trajectory')

ds

<xarray.Dataset> Size: 71GB
Dimensions:     (trajectory: 3800000, obs: 925)
Coordinates:
  * obs         (obs) int32 4kB 0 1 2 3 4 5 6 7 ... 918 919 920 921 922 923 924
  * trajectory  (trajectory) int64 30MB 0 1 2 3 4 ... 99996 99997 99998 99999
Data variables:
    lat         (trajectory, obs) float32 14GB dask.array<chunksize=(100000, 185), meta=np.ndarray>
    lon         (trajectory, obs) float32 14GB dask.array<chunksize=(100000, 185), meta=np.ndarray>
    time        (trajectory, obs) datetime64[ns] 28GB dask.array<chunksize=(100000, 185), meta=np.ndarray>
    z           (trajectory, obs) float32 14GB dask.array<chunksize=(100000, 185), meta=np.ndarray>
    stores      (trajectory) <U41 623MB 'Parcels_run_1234_2022-06-01T00:00:00...
Attributes:
    Conventions:            CF-1.6/CF-1.7
    feature_type:           trajectory
    ncei_template_version:  NCEI_NetCDF_Trajectory_Template_v2.0
    parcels_kernels:        JITParticleAdvectionRK4_3DCheckError
    parcels_mesh:           spherical
    parcels_version:        3.1.2

In [4]:
valid_data = ds.lat.notnull().any('trajectory').compute()
ds=ds.where(valid_data,drop=True)
ds=ds.assign(stores=ds.stores.isel(obs=0))
ds

<xarray.Dataset> Size: 56GB
Dimensions:     (trajectory: 3800000, obs: 741)
Coordinates:
  * obs         (obs) int32 3kB 0 1 2 3 4 5 6 7 ... 734 735 736 737 738 739 740
  * trajectory  (trajectory) int64 30MB 0 1 2 3 4 ... 99996 99997 99998 99999
Data variables:
    lat         (trajectory, obs) float32 11GB dask.array<chunksize=(100000, 185), meta=np.ndarray>
    lon         (trajectory, obs) float32 11GB dask.array<chunksize=(100000, 185), meta=np.ndarray>
    time        (trajectory, obs) datetime64[ns] 23GB dask.array<chunksize=(100000, 185), meta=np.ndarray>
    z           (trajectory, obs) float32 11GB dask.array<chunksize=(100000, 185), meta=np.ndarray>
    stores      (trajectory) object 30MB 'Parcels_run_1234_2022-06-01T00:00:0...
Attributes:
    Conventions:            CF-1.6/CF-1.7
    feature_type:           trajectory
    ncei_template_version:  NCEI_NetCDF_Trajectory_Template_v2.0
    parcels_kernels:        JITParticleAdvectionRK4_3DCheckError
    parcels_mesh:           spherical
    parcels_version:        3.1.2

In [5]:
Ntrajs = 100000
ds_subset = ds.isel(trajectory=np.random.choice(ds.trajectory.data, size=(Ntrajs, ), replace=False))
ds_subset

<xarray.Dataset> Size: 1GB
Dimensions:     (trajectory: 100000, obs: 741)
Coordinates:
  * obs         (obs) int32 3kB 0 1 2 3 4 5 6 7 ... 734 735 736 737 738 739 740
  * trajectory  (trajectory) int64 800kB 398 78357 985 ... 85270 82570 11500
Data variables:
    lat         (trajectory, obs) float32 296MB dask.array<chunksize=(100000, 185), meta=np.ndarray>
    lon         (trajectory, obs) float32 296MB dask.array<chunksize=(100000, 185), meta=np.ndarray>
    time        (trajectory, obs) datetime64[ns] 593MB dask.array<chunksize=(100000, 185), meta=np.ndarray>
    z           (trajectory, obs) float32 296MB dask.array<chunksize=(100000, 185), meta=np.ndarray>
    stores      (trajectory) object 800kB 'Parcels_run_1234_2022-06-01T00:00:...
Attributes:
    Conventions:            CF-1.6/CF-1.7
    feature_type:           trajectory
    ncei_template_version:  NCEI_NetCDF_Trajectory_Template_v2.0
    parcels_kernels:        JITParticleAdvectionRK4_3DCheckError
    parcels_mesh:           spherical
    parcels_version:        3.1.2

In [7]:
lat_min=ds_subset.lat.min().compute().data[()]
lon_min=ds_subset.lon.min().compute().data[()]

lat_max=ds_subset.lat.max().compute().data[()]
lon_max=ds_subset.lon.max().compute().data[()]

lat_bins = np.linspace(lat_min,lat_max,30)
lon_bins = np.linspace(lon_min,lon_max,30)

def calc_hist(ds):
    return xhist(
        ds.lon,
        ds.lat,
        bins=[lon_bins, lat_bins,],
        dim=["trajectory", ],
        bin_dim_suffix=""
    )

In [6]:
start_time = ds_subset['time'].isel(obs=0).compute()
start_season = start_time.dt.season

ds_subset['start_season'] = start_season
ds_subset

<xarray.Dataset> Size: 1GB
Dimensions:       (trajectory: 100000, obs: 741)
Coordinates:
  * obs           (obs) int32 3kB 0 1 2 3 4 5 6 ... 734 735 736 737 738 739 740
  * trajectory    (trajectory) int64 800kB 398 78357 985 ... 85270 82570 11500
Data variables:
    lat           (trajectory, obs) float32 296MB dask.array<chunksize=(100000, 185), meta=np.ndarray>
    lon           (trajectory, obs) float32 296MB dask.array<chunksize=(100000, 185), meta=np.ndarray>
    time          (trajectory, obs) datetime64[ns] 593MB dask.array<chunksize=(100000, 185), meta=np.ndarray>
    z             (trajectory, obs) float32 296MB dask.array<chunksize=(100000, 185), meta=np.ndarray>
    stores        (trajectory) object 800kB 'Parcels_run_1234_2022-06-01T00:0...
    start_season  (trajectory) <U3 1MB 'JJA' 'JJA' 'JJA' ... 'JJA' 'JJA' 'JJA'
Attributes:
    Conventions:            CF-1.6/CF-1.7
    feature_type:           trajectory
    ncei_template_version:  NCEI_NetCDF_Trajectory_Template_v2.0
    parcels_kernels:        JITParticleAdvectionRK4_3DCheckError
    parcels_mesh:           spherical
    parcels_version:        3.1.2

In [8]:
hist = ds_subset.groupby('start_season').apply(calc_hist)
hist

<xarray.DataArray 'histogram_lon_lat' (start_season: 1, obs: 741, lon: 29,
                                       lat: 29)> Size: 5MB
dask.array<getitem, shape=(1, 741, 29, 29), dtype=int64, chunksize=(1, 185, 29, 29), chunktype=numpy.ndarray>
Coordinates:
  * obs           (obs) int32 3kB 0 1 2 3 4 5 6 ... 734 735 736 737 738 739 740
  * lon           (lon) float32 116B -74.41 -72.46 -70.5 ... -21.62 -19.66
  * lat           (lat) float32 116B 0.796 1.583 2.37 ... 21.26 22.05 22.83
  * start_season  (start_season) object 8B 'JJA'

In [9]:
# histogram of number of data points weighted by volume resolution
# Note that depth is a non-uniform axis

# Create a dz variable
dz = np.diff(ds_subset.z)
dz = np.pad(dz, ((0, 0), (1, 0)), mode='edge') 
dz.shape

# dz = xr.DataArray(dz, coords= {'z':ds_subset.z}, dims=('trajectory', 'obs'))
# dz.shape

(100000, 741)

In [10]:
# weight by volume of grid cell (resolution = 5degree, 1degree=110km)
dVol = dz * ((1/12)*110e3) * ((1/12)*110e3*np.cos(ds_subset.lat*np.pi/180))
dVol

<xarray.DataArray 'lat' (trajectory: 100000, obs: 741)> Size: 296MB
dask.array<multiply, shape=(100000, 741), dtype=float32, chunksize=(100000, 185), chunktype=numpy.ndarray>
Coordinates:
  * obs         (obs) int32 3kB 0 1 2 3 4 5 6 7 ... 734 735 736 737 738 739 740
  * trajectory  (trajectory) int64 800kB 398 78357 985 ... 85270 82570 11500

In [14]:
# Note: The weights are automatically broadcast to the right size
hTSw = xhist(ds_subset.lon, ds_subset.lat, bins=[lon_bins, lat_bins], weights=dVol)
# np.log10(hTSw.T).plot(cmap='brg')
hTSw

<xarray.DataArray 'histogram_lon_lat' (lon_bin: 29, lat_bin: 29)> Size: 3kB
dask.array<sum-aggregate, shape=(29, 29), dtype=float32, chunksize=(29, 29), chunktype=numpy.ndarray>
Coordinates:
  * lon_bin  (lon_bin) float32 116B -74.41 -72.46 -70.5 ... -23.57 -21.62 -19.66
  * lat_bin  (lat_bin) float32 116B 0.796 1.583 2.37 3.157 ... 21.26 22.05 22.83

In [ ]:
ds_subset.z.max().data.compute()

In [ ]:
#averaging a variable

h_depth = (xhist(ds_subset.lon.where(~np.isnan(ds_subset.z)),
                   ds_subset.lat.where(~np.isnan(ds_subset.z)),
                   bins=[lon_bins, lat_bins],
                   weights=ds_subset.z.where(~np.isnan(ds_subset.z))*dVol)/
                xhist(ds_subset.lon.where(~np.isnan(ds_subset.z)),
                          ds_subset.lat.where(~np.isnan(ds_subset.z)),
                          bins=[lon_bins, lat_bins],
                          weights=dVol))

(h_depth.T).plot(vmin=ds_subset.z.min().data.compute(), vmax=ds_subset.z.max().data.compute())

In [ ]:
lon = ds_subset.lon.data.flatten()
lat = ds_subset.lat.data.flatten()
depth = ds_subset.z.data.flatten()

# Histograma ponderado por profundidad
z_sum, xedges, yedges = np.histogram2d(
    lat, lon,  # <- orden: lat, lon = y, x
    bins=(lat_bins, lon_bins),
    weights=depth
)

# Conteo de partículas por celda
counts, _, _ = np.histogram2d(
    lat, lon,
    bins=(lat_bins, lon_bins)
)

# Calcular profundidad promedio (evitando división por 0)
z_avg = np.divide(z_sum, counts, out=np.full_like(z_sum, np.nan), where=counts != 0)

# Graficar con pcolormesh
plt.figure(figsize=(5, 3))
plt.pcolormesh(yedges, xedges, z_avg, cmap="viridis")  # y: lon, x: lat
plt.colorbar(label="Mean Depth (m)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Average Particle Depth by Location")
plt.show()